# Finance AI Agent

This notebook builds a **LangChain + OpenAI** agent with:
- One finance tool (stock lookup via Yahoo Finance)
- **Conversation memory** for follow-up questions
- A **finance-only guard** that rejects off-topic queries
- An **interactive chat window** (Gradio) for user input

**Flow:** User question → finance check → agent (with memory) → tool if needed → reply

Before running:
1. `pip install -r requirements.txt`
2. Set `OPENAI_API_KEY` in your `.env` file
3. Select the project **venv** kernel in the notebook

## 1. Imports and environment setup

Load the OpenAI API key from `.env` so secrets never appear in the notebook.

In [ ]:
import os
import re
import uuid

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver

# Load variables from .env into the environment
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. Add it to your .env file, e.g. OPENAI_API_KEY=sk-..."
    )

print("Environment loaded successfully.")

## 2. Define the finance tool

The `@tool` decorator registers this function so the LLM can call it.
We fetch live stock data from the **Yahoo Finance chart API** via `requests` (no extra API key needed).

In [ ]:
# Browser-like header so Yahoo Finance accepts the request
YAHOO_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}


@tool
def get_stock_info(ticker: str) -> str:
    """Get current stock price and key financial metrics for a ticker symbol.

    Args:
        ticker: Stock ticker symbol, e.g. 'AAPL', 'MSFT', 'GOOGL', 'TSLA'.
    """
    symbol = ticker.strip().upper()

    try:
        response = requests.get(
            f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}",
            params={"interval": "1d", "range": "5d"},
            headers=YAHOO_HEADERS,
            timeout=10,
        )
        response.raise_for_status()

        payload = response.json()
        results = payload.get("chart", {}).get("result")
        if not results:
            return f"Could not find stock data for ticker '{symbol}'. Check the symbol and try again."

        meta = results[0].get("meta", {})
        price = meta.get("regularMarketPrice")
        if price is None:
            return f"Could not find stock data for ticker '{symbol}'. Check the symbol and try again."

        company = meta.get("longName") or meta.get("shortName") or symbol
        currency = meta.get("currency", "USD")
        previous_close = meta.get("chartPreviousClose")
        day_change = round(price - previous_close, 2) if previous_close else None
        day_change_pct = (
            round((day_change / previous_close) * 100, 2)
            if previous_close and day_change is not None
            else None
        )

        return (
            f"Company: {company} ({symbol})\n"
            f"Current price: {price} {currency}\n"
            f"Previous close: {previous_close} {currency}\n"
            f"Day change: {day_change} ({day_change_pct}%)\n"
            f"Day range: {meta.get('regularMarketDayLow')} - {meta.get('regularMarketDayHigh')} {currency}\n"
            f"52-week high: {meta.get('fiftyTwoWeekHigh')} {currency}\n"
            f"52-week low: {meta.get('fiftyTwoWeekLow')} {currency}\n"
            f"Exchange: {meta.get('fullExchangeName') or meta.get('exchangeName')}"
        )
    except Exception as error:
        return f"Error fetching data for '{symbol}': {error}"

## 3. Create the LLM, agent, and memory

- **ChatOpenAI** — connects to OpenAI's chat models.
- **create_agent** — LangChain agent that can call tools when needed.
- **InMemorySaver** — stores conversation history so follow-up questions keep context.

Each chat session uses a unique `thread_id` so messages are linked across turns.

In [ ]:
# Main LLM for the finance agent
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=OPENAI_API_KEY,
)

# In-memory checkpointer — persists messages within this notebook session
memory = InMemorySaver()

# Register the single finance tool with the agent
tools = [get_stock_info]

guardrailed_agent = create_agent(
    llm,
    tools,
    system_prompt=(
        "You are a helpful Finance AI assistant. "
        "You ONLY answer questions related to finance, investing, stocks, markets, "
        "banking, economics, and company financials. "
        "If the user asks about anything else, politely decline and remind them you "
        "can only help with finance questions. "
        "When users ask about a stock or ticker, use the get_stock_info tool. "
        "Use conversation history to resolve follow-up references like 'that company' or 'what about MSFT'. "
        "Reply in clear, friendly language. This is not financial advice."
    ),
    checkpointer=memory,
)

# Unique ID for this conversation — all turns share the same thread for memory
THREAD_ID = f"finance-chat-{uuid.uuid4().hex[:8]}"
AGENT_CONFIG = {"configurable": {"thread_id": THREAD_ID}}

print(f"Finance agent is ready. Session thread: {THREAD_ID}")

## 4. Guardrailed agent helper

Before each turn, classify the question as finance-related (YES/NO).
Off-topic questions are refused without calling the agent.
Streaming is used so the final model reply is captured cleanly.

In [ ]:
GUARDRAIL_REFUSAL = (
    "I can only answer finance-related questions. "
    "Please ask about stocks, markets, investing, company financials, "
    "banking, economics, or personal finance."
)

# Fast local checks before calling the LLM classifier
FINANCE_KEYWORDS = re.compile(
    r"\b("
    r"stock|stocks|share|shares|ticker|market|invest|investing|portfolio|"
    r"finance|financial|bank|banking|economy|economic|price|trading|trade|"
    r"compare|comparison|earnings|revenue|dividend|nasdaq|nyse|crypto|"
    r"apple|microsoft|google|amazon|tesla|nvidia|meta"
    r")\b",
    re.IGNORECASE,
)
TICKER_PATTERN = re.compile(r"\b[A-Z]{2,5}\b")
# Common English words that look like tickers when uppercased
TICKER_STOPWORDS = {
    "THE", "AND", "FOR", "ARE", "BUT", "NOT", "YOU", "ALL", "CAN", "HER",
    "WAS", "ONE", "OUR", "OUT", "DAY", "HAD", "HAS", "HIS", "HOW", "ITS",
    "MAY", "NEW", "NOW", "OLD", "SEE", "USE", "WAY", "WHO", "BOY", "DID",
    "GET", "HIM", "LET", "PUT", "SAY", "SHE", "TOO", "USE", "WHY", "YES",
    "IN", "ON", "AT", "TO", "OR", "IS", "IT", "AN", "AS", "BE", "DO", "IF",
    "MY", "NO", "OF", "SO", "UP", "US", "WE", "AM", "BY", "GO", "ME", "HE",
}


def get_recent_conversation_context(max_messages: int = 6) -> str:
    """Return recent chat turns so follow-up questions can be classified correctly."""
    try:
        snapshot = guardrailed_agent.get_state(AGENT_CONFIG)
        messages = snapshot.values.get("messages", [])
    except Exception:
        return ""

    lines = []
    for message in messages[-max_messages:]:
        content = getattr(message, "content", "")
        if not content:
            continue

        role = getattr(message, "type", "")
        if role == "human":
            lines.append(f"User: {content[:300]}")
        elif role == "ai":
            lines.append(f"Assistant: {content[:300]}")

    return "\n".join(lines)


def looks_like_finance_question(question: str) -> bool:
    """Quick keyword/ticker check for obvious finance questions."""
    if FINANCE_KEYWORDS.search(question):
        return True

    for word in TICKER_PATTERN.findall(question.upper()):
        if word not in TICKER_STOPWORDS:
            return True

    return False


def is_finance_related(question: str) -> bool:
    """Classify whether a user question is about finance (YES/NO)."""
    if looks_like_finance_question(question):
        return True

    context = get_recent_conversation_context()
    classifier_prompt = (
        "You classify user questions. Reply with exactly YES or NO.\n"
        "YES = the question is about finance, investing, stocks, markets, money, "
        "banking, economics, company financials, or personal finance.\n"
        "YES = also use this when the latest question is a follow-up to a prior "
        "finance conversation (e.g. 'How does Microsoft compare?', 'What about that stock?').\n"
        "NO = anything else (weather, sports, recipes, coding, history, trivia, etc.).\n\n"
        f"Recent conversation:\n{context or '(none)'}\n\n"
        f"Latest question: {question}\n"
        "Answer:"
    )
    result = llm.invoke([HumanMessage(content=classifier_prompt)])
    return result.content.strip().upper().startswith("YES")


def ask_guardrailed_finance_agent(user_message: str) -> str:
    """Run the guardrailed agent; refuse off-topic questions."""
    if not is_finance_related(user_message):
        return GUARDRAIL_REFUSAL

    final_content = ""
    for chunk in guardrailed_agent.stream(
        {"messages": [HumanMessage(content=user_message)]},
        config=AGENT_CONFIG,
        stream_mode="updates",
    ):
        for step, data in chunk.items():
            if step == "model":
                message = data["messages"][-1]
                if message.content and not getattr(message, "tool_calls", None):
                    final_content = message.content

    return final_content

## 5. Agent graph (Mermaid diagram)

`guardrailed_agent.get_graph()` returns the LangGraph structure behind the agent.
The diagram below shows how control flows between the **model** (LLM) and **tools** (`get_stock_info`) nodes until a final answer is produced.

In [ ]:
from IPython.display import Markdown, display

# Build Mermaid diagram from the compiled agent graph
agent_graph = guardrailed_agent.get_graph()
mermaid_diagram = agent_graph.draw_mermaid()

# Render the diagram in the notebook (supported in Cursor/VS Code and Jupyter)
display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))

# Optional: print raw Mermaid source for copy/paste into mermaid.live
print("Raw Mermaid source:\n")
print(mermaid_diagram)

## 6. Quick test (optional)

Run this cell to verify the agent, tool, memory, and finance guard work.

**Important:** Re-run cells 1–5 first so the agent, guardrail logic, and graph are loaded.

In [ ]:
# Test 1: Finance question — should call the stock tool
q1 = "What is the current stock price of Apple (AAPL)?"
print(f"You: {q1}")
print(f"Agent: {ask_guardrailed_finance_agent(q1)}\n")

# Test 2: Follow-up — should use memory from the previous turn
q2 = "How does Microsoft compare? Use MSFT."
print(f"You: {q2}")
print(f"Agent: {ask_guardrailed_finance_agent(q2)}\n")

# Test 3: Non-finance question — should be rejected
q3 = "What's the weather in London?"
print(f"You: {q3}")
print(f"Agent: {ask_guardrailed_finance_agent(q3)}")

## 7. Interactive chat window

Run the cell below to open the chat UI **inside the notebook output**.

Uses **Gradio** (works reliably in Cursor/VS Code notebooks). Type a finance question and press **Enter** or click **Submit**. Type `quit` to end the session.

> If you don't see the chat UI, run `%pip install gradio` in a cell, restart the kernel, then re-run all cells.

In [ ]:
import gradio as gr

CHAT_STOPPED = {"value": False}


def finance_chat_respond(message: str, history: list) -> str:
    """Handle one user message and return the agent reply for Gradio."""
    message = message.strip()
    if not message:
        return "Please enter a finance question."

    if message.lower() in {"quit", "exit"}:
        CHAT_STOPPED["value"] = True
        return "Goodbye! Refresh or re-run this cell to start a new chat."

    if CHAT_STOPPED["value"]:
        return "Chat ended. Re-run this cell to start again."

    try:
        return ask_guardrailed_finance_agent(message)
    except Exception as error:
        return f"Error: {error}"


chat_ui = gr.ChatInterface(
    fn=finance_chat_respond,
    title="Guardrailed Finance Agent",
    description=(
        "Finance questions only (stocks, markets, investing). "
        "Off-topic questions are refused. Follow-up questions use chat memory. "
        "Type **quit** to stop."
    ),
    examples=[
        "What is the current stock price of Apple (AAPL)?",
        "How does Microsoft compare? Use MSFT.",
        "What's the weather in London?",
    ],
)

# inline=True embeds the chat window directly in the notebook cell output
chat_ui.launch(inline=True, share=False, show_error=True)